In [1]:
dataset_path = "ekegusii_dataset_final.csv"

In [2]:
import shutil
import os

zip_path = "nllb-ekegusii-final.zip"
extract_to = "nllb-guz-baseline"

shutil.unpack_archive(zip_path, extract_to)
print("Unzipped to:", extract_to)

for f in os.listdir(extract_to):
    print(" -", f)

Unzipped to: nllb-guz-baseline
 - tokenizer.json
 - generation_config.json
 - model.safetensors
 - tokenizer_config.json
 - config.json


# English → Ekegusii Machine Translation using NLLB

## Objective

The objective of this notebook is to fine-tune Meta's No Language Left Behind (NLLB) model for translating Public Service Announcement (PSA) messages from English to Ekegusii.

This notebook forms part of the multilingual PSA translation project, where each team member is responsible for training a different language pair. This notebook specifically focuses on English → Ekegusii translation.

The workflow includes:

- Loading and preprocessing the dataset
- Preparing the data for NLLB
- Fine-tuning the pretrained model
- Evaluating translation quality
- Saving the trained model for inference

Model:
- facebook/nllb-200-distilled-600M

Source Language:
- English

Target Language:
- Ekegusii

In [3]:
#Step 2:Install required libraries

!pip install -q transformers datasets evaluate sacrebleu sentencepiece accelerate

# Step 3: Import Required Libraries

In this step, we import the Python libraries required for the machine translation pipeline.

The main libraries used are:

- **PyTorch**: Used for deep learning operations and checking GPU availability.
- **Pandas and NumPy**: Used for loading and analyzing the dataset.
- **Transformers**: Provides access to the NLLB tokenizer and translation model.
- **Datasets**: Used for preparing data in a format compatible with Hugging Face training tools.

Importing these libraries before model preparation ensures that the training environment is correctly configured.

In [5]:
# Data processing libraries
import pandas as pd
import numpy as np

# Deep learning framework
import torch

# Hugging Face libraries for NLP models and datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

from datasets import Dataset

# Evaluation library
import evaluate

# Utility libraries
import os
import warnings

# Hide unnecessary warning messages
warnings.filterwarnings("ignore")


# Confirm successful imports
print("Libraries imported successfully.")
print("PyTorch version:", torch.__version__)

Libraries imported successfully.
PyTorch version: 2.13.0+cu130


# Step 4: Checking GPU Availability

Deep learning models such as NLLB require significant computational resources.
Before starting model training, we verify that a GPU accelerator has been assigned.

Using a GPU reduces training time significantly compared to CPU execution.

This step confirms:
- Whether CUDA is available
- The GPU model being used
- That the environment is ready for model training

In [6]:
# Check whether a CUDA-enabled GPU is available
if torch.cuda.is_available():
    
    # Display the GPU name assigned by Kaggle
    print("GPU detected:", torch.cuda.get_device_name(0))

else:
    print("No GPU detected. Please enable GPU acceleration.")

GPU detected: NVIDIA A100-SXM4-80GB


# Step 5: Loading the Ekegusii Translation Dataset

In this step, we load the Ekegusii parallel dataset into a pandas DataFrame.

A machine translation model requires aligned sentence pairs, where each source sentence corresponds to its correct translation.

For this project, the expected translation direction is:

**Source Language:** English  
**Target Language:** Ekegusii

Before training the NLLB model, we will inspect:

- Dataset size
- Column names
- Sample translation pairs
- Missing values

This helps us identify possible data quality issues before preprocessing.

In [7]:

# Load the dataset
df = pd.read_csv(dataset_path)

# Display the first five rows
df.head()

,PSA_Id,Domain,English,Kiswahili,Ekegusii,Class,Source,English_clean,Kiswahili_clean,Ekegusii_clean,...,is_boilerplate,English_clean_n_words,English_clean_n_tokens,Kiswahili_clean_n_words,Kiswahili_clean_n_tokens,Ekegusii_clean_n_words,Ekegusii_clean_n_tokens,English_clean_tokens_str,Kiswahili_clean_tokens_str,Ekegusii_clean_tokens_str
0,1,Education,Comprehensive COVID-19 health and safety proto...,Itifaki kamili za afya na usalama za COVID-19 ...,Ase oboikeranu amachiko obochenu igoro yendwar...,PSA,original_baseline_dataset,Comprehensive COVID-19 health and safety proto...,Itifaki kamili za afya na usalama za COVID-19 ...,Ase oboikeranu amachiko obochenu igoro yendwar...,...,False,15,24,24,46,23,68,Comprehensive CO ##VI ##D - 19 health and safe...,It ##ifa ##ki kami ##li za af ##ya na usa ##la...,As ##e ob ##oi ##kera ##nu amach ##iko ob ##oc...
1,2,Education,Digital learning platform providing free educa...,Jukwaa la kujifunza kidijitali linalotoa maudh...,Omoreberio bwamasomo bwechisemi o'Kenya goeter...,PSA,original_baseline_dataset,Digital learning platform providing free educa...,Jukwaa la kujifunza kidijitali linalotoa maudh...,Omoreberio bwamasomo bwechisemi o'Kenya goeter...,...,False,14,21,23,52,19,57,Digital learning platform providing free educa...,Ju ##kwa ##a la ku ##ji ##fu ##nza ki ##dij ##...,Om ##ore ##beri ##o b ##wa ##mas ##omo b ##we ...
2,3,Education,KUCCPS portal will open in March 2025 for univ...,Lango la KUCCPS litafunguliwa Machi 2025 kwa n...,Eintaneti yekeombe keria ekenen gekorangeria a...,PSA,original_baseline_dataset,KUCCPS portal will open in March 2025 for univ...,Lango la KUCCPS litafunguliwa Machi 2025 kwa n...,Eintaneti yekeombe keria ekenen gekorangeria a...,...,False,17,22,21,40,31,88,K ##UC ##CP ##S portal will open in March 2025...,Lang ##o la K ##UC ##CP ##S lit ##af ##ung ##u...,Ein ##tane ##ti ye ##ke ##omb ##e ker ##ia ek ...
3,4,Education,Target to increase school feeding beneficiarie...,Lengo ni kuongeza wanufaika wa chakula shuleni...,Norengete kogesa abana bakoyeria nechisukuru k...,PSA,original_baseline_dataset,Target to increase school feeding beneficiarie...,Lengo ni kuongeza wanufaika wa chakula shuleni...,Norengete kogesa abana bakoyeria nechisukuru k...,...,False,16,20,19,32,15,36,Target to increase school feeding bene ##fici ...,Len ##go ni ku ##onge ##za wa ##nu ##fa ##ika ...,Nor ##eng ##ete ko ##ges ##a ab ##ana bak ##oy...
4,5,Education,Launch of inclusive education programs with as...,Uzinduzi wa programu za elimu jumuishi zenye t...,Omoroberio bwogochaka chisemi ase bonsi goeter...,PSA,original_baseline_dataset,Launch of inclusive education programs with as...,Uzinduzi wa programu za elimu jumuishi zenye t...,Omoroberio bwogochaka chisemi ase bonsi goeter...,...,False,12,16,13,29,18,50,Launch of inclusive education programs with as...,Uz ##ind ##uzi wa programu za eli ##mu ju ##mu...,Om ##oro ##beri ##o b ##wo ##go ##cha ##ka chi...


# Step 6: Exploring the Multilingual Dataset

This dataset contains multilingual parallel PSA sentences in three languages:

- English
- Kiswahili
- Ekegusii

The objective is to fine-tune NLLB for bidirectional translation:

1. English ↔ Ekegusii
2. Kiswahili ↔ Ekegusii

Since NLLB supports multilingual translation, the training dataset will later be expanded by creating reversed translation pairs.

For example:

English → Ekegusii

will also create:

Ekegusii → English

This allows one model to learn translation in multiple directions.

In [8]:
# Check dataset size
print("Dataset shape:")
print(df.shape)


# Display the important language columns
language_columns = [
    "English_clean",
    "Kiswahili_clean",
    "Ekegusii_clean"
]

print("\nLanguage columns:")
print(language_columns)


# Check missing values only for translation columns
print("\nMissing values:")
print(df[language_columns].isnull().sum())


# Display sample translation pairs
df[language_columns].head()

Dataset shape:
(5126, 24)

Language columns:
['English_clean', 'Kiswahili_clean', 'Ekegusii_clean']

Missing values:
English_clean      0
Kiswahili_clean    0
Ekegusii_clean     0
dtype: int64


,English_clean,Kiswahili_clean,Ekegusii_clean
0,Comprehensive COVID-19 health and safety proto...,Itifaki kamili za afya na usalama za COVID-19 ...,Ase oboikeranu amachiko obochenu igoro yendwar...
1,Digital learning platform providing free educa...,Jukwaa la kujifunza kidijitali linalotoa maudh...,Omoreberio bwamasomo bwechisemi o'Kenya goeter...
2,KUCCPS portal will open in March 2025 for univ...,Lango la KUCCPS litafunguliwa Machi 2025 kwa n...,Eintaneti yekeombe keria ekenen gekorangeria a...
3,Target to increase school feeding beneficiarie...,Lengo ni kuongeza wanufaika wa chakula shuleni...,Norengete kogesa abana bakoyeria nechisukuru k...
4,Launch of inclusive education programs with as...,Uzinduzi wa programu za elimu jumuishi zenye t...,Omoroberio bwogochaka chisemi ase bonsi goeter...


# Handling an Unsupported NLLB Language

Ekegusii is not included among the original NLLB-200 supported languages.

To adapt NLLB for Ekegusii translation, the language identifier `guz_Latn` is introduced as a new language token during fine-tuning.

Because Kiswahili is a supported NLLB language and belongs to the same Bantu language family as Ekegusii, it is included as a transfer language to provide additional linguistic information.

The training data therefore includes four translation directions:

- English → Ekegusii
- Ekegusii → English
- Kiswahili → Ekegusii
- Ekegusii → Kiswahili

# Step 7: Verify NLLB Language Code Support

NLLB-200 was pretrained on a fixed set of languages. Before fine-tuning the model for Ekegusii, we need to verify whether the language identifier `guz_Latn` is already available in the NLLB tokenizer.

This step is important because:

- If `guz_Latn` exists, we can directly use it during training.
- If `guz_Latn` is missing, we need to extend the tokenizer vocabulary and model embeddings to introduce Ekegusii as a new language identifier.

This follows the requirement of adapting NLLB for a low-resource language that is not originally included in the model.

In [8]:
# Check whether guz_Latn exists in the tokenizer vocabulary

ekegusii_code = "guz_Latn"

# Search tokenizer vocabulary
vocab = tokenizer.get_vocab()

if ekegusii_code in vocab:
    print("guz_Latn exists in tokenizer vocabulary.")
    print("Token ID:", vocab[ekegusii_code])

else:
    print("guz_Latn is NOT present in tokenizer vocabulary.")

NameError: name 'tokenizer' is not defined

# Step 8: Extending NLLB for Ekegusii

Since Ekegusii is not included in the original NLLB-200 language inventory, we introduce a new language identifier:

`guz_Latn`

This allows the model to distinguish Ekegusii sentences during training.

The process involves:

1. Adding `guz_Latn` as a special tokenizer token.
2. Increasing the model embedding size to include the new token.
3. Initializing the new embedding so the model can learn Ekegusii representations during fine-tuning.

This approach adapts an existing multilingual model to a low-resource language without training a model from scratch.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Define the pretrained NLLB model
model_name = "facebook/nllb-200-distilled-600M"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define Ekegusii language token
ekegusii_token = "guz_Latn"

# Add the token if it does not already exist
if ekegusii_token not in tokenizer.get_vocab():

    tokenizer.add_special_tokens(
        {"additional_special_tokens": [ekegusii_token]}
    )

    print("Added guz_Latn to tokenizer.")

else:
    print("guz_Latn already exists.")

# Check new vocabulary size
print("Tokenizer vocabulary size:", len(tokenizer))

# Step 8: Loading NLLB Model and Updating Embeddings

The tokenizer has been extended with the new Ekegusii language identifier:

`guz_Latn`

However, the pretrained NLLB model was originally created with the old vocabulary size. The model's embedding layer must therefore be updated to include the newly added token.

This step performs the following:

1. Loads the pretrained NLLB-200 distilled model.
2. Resizes the model embeddings to match the updated tokenizer.
3. Verifies that the tokenizer and model vocabulary sizes are aligned.

After this step, the model architecture is ready to learn Ekegusii during fine-tuning.

In [ ]:
# Load the pretrained NLLB sequence-to-sequence model

from transformers import AutoModelForSeq2SeqLM

# Load the model using the same checkpoint as the tokenizer
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


# Resize the model embeddings to include the new guz_Latn token
model.resize_token_embeddings(len(tokenizer))


# Verify tokenizer and model vocabulary sizes match
print("Tokenizer vocabulary size:", len(tokenizer))
print(
    "Model embedding vocabulary size:",
    model.get_input_embeddings().num_embeddings
)

# Step 9: Creating the Bidirectional Translation Dataset

The original dataset contains aligned sentences in three languages:

- English
- Kiswahili
- Ekegusii

Since the objective is to train a single NLLB model for multiple translation directions, we expand the dataset by creating four translation tasks:

1. English → Ekegusii
2. Ekegusii → English
3. Kiswahili → Ekegusii
4. Ekegusii → Kiswahili

Each example will contain:

- `source_text`: The input sentence
- `target_text`: The expected translation
- `source_lang`: NLLB source language identifier
- `target_lang`: NLLB target language identifier

The language identifiers used are:

| Language | NLLB Code |
|----------|-----------|
| English | `eng_Latn` |
| Kiswahili | `swh_Latn` |
| Ekegusii | `guz_Latn` |

This expanded dataset allows NLLB to learn both translation directions while using the shared linguistic knowledge between English, Kiswahili, and Ekegusii.

In [9]:
# Create English → Ekegusii translation examples

eng_to_eke = pd.DataFrame({
    "source_text": df["English_clean"],
    "target_text": df["Ekegusii_clean"],
    "source_lang": "eng_Latn",
    "target_lang": "guz_Latn"
})


# Create Ekegusii → English translation examples

eke_to_eng = pd.DataFrame({
    "source_text": df["Ekegusii_clean"],
    "target_text": df["English_clean"],
    "source_lang": "guz_Latn",
    "target_lang": "eng_Latn"
})


# Create Kiswahili → Ekegusii translation examples

swh_to_eke = pd.DataFrame({
    "source_text": df["Kiswahili_clean"],
    "target_text": df["Ekegusii_clean"],
    "source_lang": "swh_Latn",
    "target_lang": "guz_Latn"
})


# Create Ekegusii → Kiswahili translation examples

eke_to_swh = pd.DataFrame({
    "source_text": df["Ekegusii_clean"],
    "target_text": df["Kiswahili_clean"],
    "source_lang": "guz_Latn",
    "target_lang": "swh_Latn"
})


# Combine all translation directions into one dataset

translation_df = pd.concat(
    [
        eng_to_eke,
        eke_to_eng,
        swh_to_eke,
        eke_to_swh
    ],
    ignore_index=True
)


# Display dataset information

print("New dataset shape:")
print(translation_df.shape)


# Display first examples

translation_df.head()

New dataset shape:
(20504, 4)


,source_text,target_text,source_lang,target_lang
0,Comprehensive COVID-19 health and safety proto...,Ase oboikeranu amachiko obochenu igoro yendwar...,eng_Latn,guz_Latn
1,Digital learning platform providing free educa...,Omoreberio bwamasomo bwechisemi o'Kenya goeter...,eng_Latn,guz_Latn
2,KUCCPS portal will open in March 2025 for univ...,Eintaneti yekeombe keria ekenen gekorangeria a...,eng_Latn,guz_Latn
3,Target to increase school feeding beneficiarie...,Norengete kogesa abana bakoyeria nechisukuru k...,eng_Latn,guz_Latn
4,Launch of inclusive education programs with as...,Omoroberio bwogochaka chisemi ase bonsi goeter...,eng_Latn,guz_Latn


# Step 10: Checking Translation Direction Distribution

Before splitting the dataset for training, we verify that all translation directions are represented equally.

A balanced dataset prevents the model from learning one translation direction better than the others.

We check the combination of:

- Source language
- Target language

Expected distribution:

- `eng_Latn → guz_Latn`: 5,126 examples
- `guz_Latn → eng_Latn`: 5,126 examples
- `swh_Latn → guz_Latn`: 5,126 examples
- `guz_Latn → swh_Latn`: 5,126 examples

In [10]:
# Count examples for each translation direction

direction_distribution = (
    translation_df
    .groupby(["source_lang", "target_lang"])
    .size()
    .reset_index(name="count")
)


# Display distribution

direction_distribution

,source_lang,target_lang,count
0,eng_Latn,guz_Latn,5126
1,guz_Latn,eng_Latn,5126
2,guz_Latn,swh_Latn,5126
3,swh_Latn,guz_Latn,5126


# Step 11: Dataset Quality Check

Before training the NLLB model, we perform final quality checks on the expanded translation dataset.

The checks include:

- Missing values
- Duplicate translation pairs
- Empty sentences

Clean parallel data is important because machine translation models learn directly from the provided sentence pairs. Noisy or repeated examples can reduce translation quality.

In [11]:
# Check missing values

print("Missing values:")
print(translation_df.isnull().sum())


# Check duplicate translation pairs

duplicates = translation_df.duplicated(
    subset=["source_text", "target_text"]
).sum()

print("\nDuplicate translation pairs:")
print(duplicates)


# Check empty sentences

empty_source = (
    translation_df["source_text"]
    .str.strip()
    .eq("")
    .sum()
)

empty_target = (
    translation_df["target_text"]
    .str.strip()
    .eq("")
    .sum()
)

print("\nEmpty source sentences:", empty_source)
print("Empty target sentences:", empty_target)

Missing values:
source_text    0
target_text    0
source_lang    0
target_lang    0
dtype: int64

Duplicate translation pairs:
269

Empty source sentences: 0
Empty target sentences: 0


# Step 12: Removing Duplicate Translation Pairs

Duplicate sentence pairs can cause a translation model to overfit by repeatedly seeing the same examples.

Although some repetition is expected in PSA datasets, we remove exact duplicate source-target pairs to improve dataset diversity.

Only identical translation pairs are removed. Similar sentences with different meanings are retained.

In [12]:
# Store size before removing duplicates

before_duplicates = len(translation_df)


# Remove duplicate translation pairs

translation_df = translation_df.drop_duplicates(
    subset=["source_text", "target_text"]
).reset_index(drop=True)


# Store size after cleaning

after_duplicates = len(translation_df)


# Display results

print("Dataset size before removing duplicates:", before_duplicates)
print("Dataset size after removing duplicates:", after_duplicates)

print(
    "Duplicates removed:",
    before_duplicates - after_duplicates
)

Dataset size before removing duplicates: 20504
Dataset size after removing duplicates: 20235
Duplicates removed: 269


# Step 13: Splitting the Dataset into Train, Validation, and Test Sets

Machine translation models require separate datasets for:

1. Training set:
   - Used by the model to learn translation patterns.

2. Validation set:
   - Used during training to monitor performance and detect overfitting.

3. Test set:
   - Used after training to evaluate the final translation ability on unseen examples.

The cleaned dataset is divided using the following proportions:

- 80% Training
- 10% Validation
- 10% Testing

The split is performed randomly while preserving the multilingual translation examples.

In [13]:
!pip install -q scikit-learn

In [14]:
from sklearn.model_selection import train_test_split


# First split:
# 80% training and 20% temporary data

train_df, temp_df = train_test_split(
    translation_df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)


# Second split:
# Divide the remaining 20% into validation and test sets

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    shuffle=True
)


# Display dataset sizes

print("Training set size:", len(train_df))
print("Validation set size:", len(val_df))
print("Test set size:", len(test_df))

Training set size: 16188
Validation set size: 2023
Test set size: 2024


# Step 14: Checking Translation Direction Distribution After Splitting

After dividing the dataset into training, validation, and test sets, we verify that all translation directions are still represented.

This ensures that the model is trained and evaluated on all required tasks:

- English → Ekegusii
- Ekegusii → English
- Kiswahili → Ekegusii
- Ekegusii → Kiswahili

Maintaining balanced language directions prevents biased evaluation results.

In [15]:
# Function to display translation direction distribution

def check_direction_distribution(data, name):
    
    print(f"\n{name} distribution:")
    
    distribution = (
        data
        .groupby(["source_lang", "target_lang"])
        .size()
        .reset_index(name="count")
    )
    
    display(distribution)


# Check all splits

check_direction_distribution(train_df, "Training")
check_direction_distribution(val_df, "Validation")
check_direction_distribution(test_df, "Test")


Training distribution:


,source_lang,target_lang,count
0,eng_Latn,guz_Latn,4036
1,guz_Latn,eng_Latn,3941
2,guz_Latn,swh_Latn,4112
3,swh_Latn,guz_Latn,4099



Validation distribution:


,source_lang,target_lang,count
0,eng_Latn,guz_Latn,489
1,guz_Latn,eng_Latn,516
2,guz_Latn,swh_Latn,507
3,swh_Latn,guz_Latn,511



Test distribution:


,source_lang,target_lang,count
0,eng_Latn,guz_Latn,500
1,guz_Latn,eng_Latn,525
2,guz_Latn,swh_Latn,495
3,swh_Latn,guz_Latn,504


# Step 15: Converting DataFrames to Hugging Face Dataset Format

The Hugging Face `datasets` library provides an efficient format for training transformer models.

The current data is stored as pandas DataFrames. We convert:

- Training data
- Validation data
- Test data

into Hugging Face Dataset objects.

This format allows us to:
- efficiently tokenize sentences
- process data in batches
- integrate with the NLLB training pipeline

In [16]:
from datasets import Dataset, DatasetDict


# Convert pandas DataFrames into Hugging Face datasets

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)


# Combine into a DatasetDict

dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})


# Display dataset structure

dataset

DatasetDict({
    train: Dataset({
        features: ['source_text', 'target_text', 'source_lang', 'target_lang', '__index_level_0__'],
        num_rows: 16188
    })
    validation: Dataset({
        features: ['source_text', 'target_text', 'source_lang', 'target_lang', '__index_level_0__'],
        num_rows: 2023
    })
    test: Dataset({
        features: ['source_text', 'target_text', 'source_lang', 'target_lang', '__index_level_0__'],
        num_rows: 2024
    })
})

# Step 16: Removing Unnecessary Dataset Columns

When converting from pandas to Hugging Face Dataset format, an additional column called:

`__index_level_0__`

was created.

This column only stores the original pandas row index and does not contain translation information.

Removing unnecessary columns reduces memory usage and keeps the dataset focused on the information required for NLLB training:

- source sentence
- target sentence
- source language
- target language

In [17]:
# Remove the automatically created pandas index column

dataset = dataset.remove_columns(
    ["__index_level_0__"]
)


# Display updated structure

dataset

DatasetDict({
    train: Dataset({
        features: ['source_text', 'target_text', 'source_lang', 'target_lang'],
        num_rows: 16188
    })
    validation: Dataset({
        features: ['source_text', 'target_text', 'source_lang', 'target_lang'],
        num_rows: 2023
    })
    test: Dataset({
        features: ['source_text', 'target_text', 'source_lang', 'target_lang'],
        num_rows: 2024
    })
})

# Step 17: Preparing NLLB Tokenization

Transformer models cannot process raw text directly. The sentences must first be converted into numerical token representations.

For NLLB, tokenization requires specifying:

1. Source language:
   - The language of the input sentence.

2. Target language:
   - The language the model should generate.

The dataset contains four translation directions:

- English → Ekegusii
- Ekegusii → English
- Kiswahili → Ekegusii
- Ekegusii → Kiswahili

Therefore, the tokenizer language settings must change dynamically for every example.

The tokenizer will generate:

- `input_ids`: numerical representation of the source sentence
- `attention_mask`: identifies real tokens from padding
- `labels`: numerical representation of the target translation

In [18]:
# Take one example from the dataset

sample = dataset["train"][0]


# Display the example

print("Source language:", sample["source_lang"])
print("Target language:", sample["target_lang"])

print("\nSource text:")
print(sample["source_text"])

print("\nTarget text:")
print(sample["target_text"])

Source language: swh_Latn
Target language: guz_Latn

Source text:
Madarasa ya mahitaji maalum yameongezwa hadi 500 – yanaomba uhamisho kabla ya Desemba 20!

Target text:
Ebirasi biogosomera bia abana baria nokogania kwabeene mbiagesegete goika 500, toma ogosaba gwokong'anya nyuma yechitariki 20 chiomotienyi 12!


# Step 18: Selecting Correct Language Samples for Tokenization Analysis

The dataset contains mixed translation directions, therefore selecting the first row does not guarantee a specific language.

To correctly analyze tokenization behavior, we explicitly search for examples based on their language identifiers.

We extract:

- English sentence (`eng_Latn`)
- Kiswahili sentence (`swh_Latn`)
- Ekegusii sentence (`guz_Latn`)

These samples will then be passed through the NLLB tokenizer.

In [ ]:
# Initialize empty samples

english_sample = None
swahili_sample = None
ekegusii_sample = None


# Search the dataset for each language

for example in dataset["train"]:
    
    if english_sample is None:
        if example["source_lang"] == "eng_Latn":
            english_sample = example["source_text"]
            
    if swahili_sample is None:
        if example["source_lang"] == "swh_Latn":
            swahili_sample = example["source_text"]
            
    if ekegusii_sample is None:
        if example["source_lang"] == "guz_Latn":
            ekegusii_sample = example["source_text"]
            
    if english_sample and swahili_sample and ekegusii_sample:
        break


print("English sample:")
print(english_sample)

print("\nKiswahili sample:")
print(swahili_sample)

print("\nEkegusii sample:")
print(ekegusii_sample)

# Step 18.1: Comparing Subword Tokenization Across Languages

NLLB uses a SentencePiece tokenizer that converts text into smaller subword units.

The original NLLB model was not trained with Ekegusii as a supported language. Therefore, Ekegusii words are expected to be represented using existing multilingual subword pieces rather than dedicated Ekegusii tokens.

This comparison examines:

- English token representation
- Kiswahili token representation
- Ekegusii token representation

The results help explain how the model will learn Ekegusii during fine-tuning.

In [ ]:
# Tokenize each language sample

english_tokens = tokenizer.tokenize(english_sample)

swahili_tokens = tokenizer.tokenize(swahili_sample)

ekegusii_tokens = tokenizer.tokenize(ekegusii_sample)


# Display tokens and token counts

print("English tokens:")
print(english_tokens)
print("Number of English tokens:", len(english_tokens))


print("\nKiswahili tokens:")
print(swahili_tokens)
print("Number of Kiswahili tokens:", len(swahili_tokens))


print("\nEkegusii tokens:")
print(ekegusii_tokens)
print("Number of Ekegusii tokens:", len(ekegusii_tokens))

Since Ekegusii is not included as an original NLLB-200 language, its sentences are represented through existing multilingual subword units rather than language-specific vocabulary. Tokenization analysis showed increased subword fragmentation compared to English and Kiswahili, highlighting the low-resource adaptation challenge.

# Step 19: Tokenizing the Complete Dataset for NLLB Training

The NLLB model cannot process raw text directly. All sentences must be converted into numerical token representations.

For every example, we create:

- `input_ids`
    - Numerical representation of the source sentence.

- `attention_mask`
    - Indicates which tokens are actual text and which are padding.

- `labels`
    - Numerical representation of the target translation.

Because our dataset contains multiple translation directions, the tokenizer dynamically changes:

- Source language (`src_lang`)
- Target language (`tgt_lang`)

according to each example.

This ensures that NLLB correctly learns:

- English → Ekegusii
- Ekegusii → English
- Kiswahili → Ekegusii
- Ekegusii → Kiswahili

In [ ]:
# Maximum number of tokens allowed per sentence

max_length = 128


def preprocess_function(example):
    
    # Set the source language dynamically
    tokenizer.src_lang = example["source_lang"]
    
    
    # Tokenize input sentence
    model_inputs = tokenizer(
        example["source_text"],
        max_length=max_length,
        truncation=True
    )
    
    
    # Set target language dynamically
    tokenizer.tgt_lang = example["target_lang"]
    
    
    # Tokenize target sentence
    labels = tokenizer(
        text_target=example["target_text"],
        max_length=max_length,
        truncation=True
    )
    
    
    # Store target token IDs as labels
    model_inputs["labels"] = labels["input_ids"]
    
    
    return model_inputs

In [ ]:
# Test preprocessing on one example

test_token = preprocess_function(dataset["train"][0])


print(test_token.keys())


print("\nInput IDs length:")
print(len(test_token["input_ids"]))


print("\nLabel length:")
print(len(test_token["labels"]))

# Step 19.1: Applying Tokenization to the Full Dataset

The tokenization function is now applied to all dataset splits:

- Training
- Validation
- Test

The original text columns are removed after tokenization because the model only requires:

- `input_ids`
- `attention_mask`
- `labels`

Removing the text columns reduces memory usage during training.

In [ ]:
# Apply tokenization to the complete dataset

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=False
)


# Remove unnecessary text columns

tokenized_dataset = tokenized_dataset.remove_columns(
    [
        "source_text",
        "target_text",
        "source_lang",
        "target_lang"
    ]
)


# Display the final structure

tokenized_dataset

# Step 20: Preparing the Data Collator

During training, examples in a batch usually have different sentence lengths.

For example:

Example 1:
- 20 tokens

Example 2:
- 45 tokens

Example 3:
- 70 tokens


The model requires all examples in a batch to have the same length.

The Data Collator automatically:

- Pads shorter sequences
- Creates batches
- Pads labels correctly
- Replaces padding tokens in labels with `-100`

The value `-100` tells PyTorch to ignore these positions when calculating the loss.

This allows efficient GPU training.

In [ ]:
from transformers import DataCollatorForSeq2Seq


# Create the data collator

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)


print("Data collator created successfully.")

In [ ]:
# Create a small batch from training examples

batch = data_collator(
    [
        tokenized_dataset["train"][0],
        tokenized_dataset["train"][1],
        tokenized_dataset["train"][2]
    ]
)


# Display batch information

for key, value in batch.items():
    print(key, value.shape)

# Step 21: Configuring NLLB Fine-Tuning

The NLLB model is fine-tuned using the Hugging Face Trainer API.

The training configuration controls:

- Learning rate
- Number of training epochs
- Batch size
- Evaluation frequency
- Checkpoint saving
- Mixed precision training

Because NLLB-200 distilled 600M is a large transformer model, memory-efficient settings are used to fit training on a Kaggle Tesla T4 GPU.

Mixed precision (`fp16`) reduces GPU memory usage and speeds up training.

# Step 21.1: Checking Hugging Face Transformers Version

Before creating the training configuration, we check the installed Transformers version.

The Trainer API changes slightly between versions, so verifying the version helps us use compatible training arguments and avoid configuration errors.

In [ ]:
import transformers

print("Transformers version:")
print(transformers.__version__)

# Step 21.2: Creating NLLB Fine-Tuning Training Arguments

The training configuration defines how the NLLB model will be fine-tuned.

Important settings:

- `learning_rate`
    - Controls how much the pretrained model weights are updated.

- `per_device_train_batch_size`
    - Number of examples processed by the GPU at once.

- `gradient_accumulation_steps`
    - Simulates a larger batch size without exceeding GPU memory.

- `fp16`
    - Enables mixed precision training for faster computation and lower memory usage on the Tesla T4 GPU.

- `num_train_epochs`
    - Number of times the model sees the training dataset.

The selected values are conservative because NLLB-600M is a large model and Ekegusii is a low-resource language requiring careful fine-tuning.

In [ ]:
from transformers import Seq2SeqTrainingArguments


training_args = Seq2SeqTrainingArguments(

    # Directory where checkpoints will be saved
    output_dir="./nllb-ekegusii",

    # Training settings
    num_train_epochs=3,
    learning_rate=2e-5,

    # Batch size settings for Tesla T4
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    # Gradient accumulation helps fit larger effective batches
    gradient_accumulation_steps=4,

    # Evaluation and checkpoint saving
    eval_strategy="epoch",
    save_strategy="epoch",

    # Keep best checkpoint
    load_best_model_at_end=True,

    # Use fp16 mixed precision on Tesla T4
    fp16=True,

    # Logging
    logging_steps=100,

    # Prevent excessive checkpoint storage
    save_total_limit=2,

    # Reporting
    report_to="none",

    # Generation settings for evaluation
    predict_with_generate=True
)


print("Training arguments created successfully.")

# Step 22: Creating the Seq2Seq Trainer

The Hugging Face `Seq2SeqTrainer` combines all components required for fine-tuning:

- Pretrained NLLB model
- Tokenized training dataset
- Tokenized validation dataset
- Training configuration
- Data collator

During training, the trainer will:

1. Feed tokenized sentences into NLLB.
2. Compare generated translations with the reference translations.
3. Calculate the translation loss.
4. Update model parameters.
5. Evaluate performance on the validation set.

This allows the pretrained multilingual model to adapt specifically to Ekegusii translation tasks.

In [ ]:
from transformers import Seq2SeqTrainer


# Create the trainer

trainer = Seq2SeqTrainer(
    
    # The NLLB model
    model=model,
    
    # Training configuration
    args=training_args,
    
    # Training dataset
    train_dataset=tokenized_dataset["train"],
    
    # Validation dataset
    eval_dataset=tokenized_dataset["validation"],
    
    # Tokenizer for processing generated sequences
    processing_class=tokenizer,
    
    # Dynamic padding
    data_collator=data_collator
)


print("Seq2Seq Trainer created successfully.")

# Step 23: Verifying the Training Environment

Before starting fine-tuning, we verify that:

1. CUDA GPU is available.
2. The NLLB model is loaded on the correct device.
3. Mixed precision training can be used with the Tesla T4 GPU.

Using the GPU is essential because NLLB-200 distilled 600M is too large for practical CPU training.

In [ ]:
import torch


# Check GPU availability

if torch.cuda.is_available():
    
    print("GPU detected:")
    print(torch.cuda.get_device_name(0))
    
    print("\nGPU memory allocated:")
    print(
        round(torch.cuda.memory_allocated(0) / 1024**3, 2),
        "GB"
    )
    
else:
    print("No GPU detected")

In [ ]:
# Check where the model parameters are located

print(
    "Model device:",
    next(model.parameters()).device
)

# Step 24: Fine-Tuning NLLB for Ekegusii Translation

The NLLB model is now fine-tuned using the prepared multilingual dataset.

The training objective is to adapt the pretrained multilingual model to perform:

1. English → Ekegusii
2. Ekegusii → English
3. Kiswahili → Ekegusii
4. Ekegusii → Kiswahili

The model learns from the translation pairs by minimizing the difference between:

- The generated translation
- The reference target translation

During training we monitor:

- Training loss
- Validation loss
- Saved checkpoints

The Tesla T4 GPU and FP16 mixed precision are used to improve training efficiency.

In [ ]:
# Start fine-tuning

training_results = trainer.train()

In [ ]:
import os

for root, dirs, files in os.walk("./nllb-ekegusii"):
    level = root.replace("./nllb-ekegusii", "").count(os.sep)
    indent = " " * 2 * level
    print(indent + os.path.basename(root) + "/")
    for f in files:
        print(indent + "  " + f)

# Step 24.1: Checking Saved Model Files

The training completed successfully, but checkpoint saving failed while saving optimizer states.

We check which model files were successfully written before continuing.

In [ ]:
import os

for root, dirs, files in os.walk("./nllb-ekegusii"):
    level = root.replace("./nllb-ekegusii", "").count(os.sep)
    
    if level < 2:
        print(root)
        for file in files:
            print("   ", file)

# Step 25: Loading the Fine-Tuned NLLB Model

After completing fine-tuning, we reload the trained checkpoint for evaluation.

The checkpoint contains the updated NLLB parameters learned from:

- English ↔ Ekegusii translation pairs
- Kiswahili ↔ Ekegusii translation pairs

Loading the model separately ensures that evaluation is performed on the saved trained model rather than the temporary training state.

We use the final checkpoint:

`checkpoint-1518`

which corresponds to the completion of the 3 training epochs.

In [ ]:
import shutil
import os

base = "./nllb-ekegusii"

for folder in ["checkpoint-506", "checkpoint-1012"]:
    path = os.path.join(base, folder)
    if os.path.exists(path):
        shutil.rmtree(path)
        print("Deleted:", path)

print("Done")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import os

checkpoint_path = "./nllb-ekegusii/checkpoint-1518"

final_path = "./nllb-ekegusii-final"

os.makedirs(final_path, exist_ok=True)


model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint_path)

tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)


model.save_pretrained(final_path)

tokenizer.save_pretrained(final_path)


print("Saved final model successfully ✅")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

test_path = "./nllb-ekegusii-final"

test_tokenizer = AutoTokenizer.from_pretrained(test_path)

test_model = AutoModelForSeq2SeqLM.from_pretrained(test_path)

print("Reload successful ✅")

In [ ]:
import shutil

zip_path = shutil.make_archive(
    "/kaggle/working/nllb-ekegusii-final",
    "zip",
    "./nllb-ekegusii-final"
)

print(zip_path)

In [ ]:
import os

file = "/kaggle/working/nllb-ekegusii-final.zip"

print(
    "Backup size:",
    round(os.path.getsize(file)/1024/1024,2),
    "MB"
)

# Step 25.1: Checking Model and Tokenizer Compatibility

Because Ekegusii (`guz_Latn`) was added manually to the tokenizer, we verify that:

- The tokenizer vocabulary size matches the model embeddings.
- The new language token is still available.

This ensures the saved checkpoint can generate translations correctly.

In [ ]:
# Check tokenizer and model vocabulary sizes

print("Tokenizer vocabulary size:",
      len(tokenizer))

print("Model embedding size:",
      model.model.shared.num_embeddings)


# Check language token

if "guz_Latn" in tokenizer.get_vocab():
    print("\nguz_Latn token exists.")
    print(
        "Token ID:",
        tokenizer.convert_tokens_to_ids("guz_Latn")
    )
else:
    print("\nguz_Latn token not found.")

# Step 26: Creating the Translation Function

A translation function is created to standardize inference.

The function performs the following steps:

1. Sets the source language.
2. Tokenizes the input sentence.
3. Generates the translation using the fine-tuned NLLB model.
4. Forces generation into the desired target language.
5. Decodes the generated tokens back into text.

This function will be used for both manual testing and automatic evaluation.

In [ ]:
def translate_text(
    text,
    source_lang,
    target_lang,
    max_length=128
):
    
    # Set source language
    tokenizer.src_lang = source_lang
    
    
    # Tokenize input
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length
    )
    
    
    # Move tensors to GPU
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }
    
    
    # Get target language token ID
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(
        target_lang
    )
    
    
    # Generate translation
    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        max_length=max_length,
        num_beams=5
    )
    
    
    # Decode output
    translation = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )[0]
    
    
    return translation


print("Translation function created successfully.")

# Step 27: Testing Translation Generation

Before automatic evaluation, we test the fine-tuned model manually.

These examples verify that:

- The model generates output.
- The target language token is working.
- The model can translate in the intended directions.

We begin with English → Ekegusii.

In [ ]:
# Test sentence: English -> Ekegusii

english_sentence = (
    "Monitor nutrition in school feeding programs 2020."
)

translation = translate_text(
    english_sentence,
    source_lang="eng_Latn",
    target_lang="guz_Latn"
)


print("English:")
print(english_sentence)

print("\nGenerated Ekegusii:")
print(translation)

# Step 27.2: Testing All Translation Directions

The trained model supports four translation directions:

1. English → Ekegusii
2. Ekegusii → English
3. Kiswahili → Ekegusii
4. Ekegusii → Kiswahili

Testing each direction helps identify whether the model learned bidirectional translation rather than only memorizing one direction.

In [ ]:
# Example sentences from the dataset

tests = [
    (
        "English → Ekegusii",
        "eng_Latn",
        "guz_Latn",
        "Monitor nutrition in school feeding programs 2020."
    ),
    
    (
        "Ekegusii → English",
        "guz_Latn",
        "eng_Latn",
        "Meministiri yamang'ana abasae, neyechisemi nkobwaterana bare ase emeroberio yogosomia amasomo egdichitari arengete baria batari gochia school."
    ),
    
    (
        "Kiswahili → Ekegusii",
        "swh_Latn",
        "guz_Latn",
        "Madarasa ya mahitaji maalum yameongezwa hadi 500 – yanaomba uhamisho kabla ya Desemba 20!"
    ),
    
    (
        "Ekegusii → Kiswahili",
        "guz_Latn",
        "swh_Latn",
        "Ebirasi biogosomera bia abana baria nokogania kwabeene mbiagesegete goika 500, toma ogosaba gwokong'anya nyuma yechitariki 20 chiomotienyi 12!"
    )
]


for name, src, tgt, sentence in tests:
    
    result = translate_text(
        sentence,
        source_lang=src,
        target_lang=tgt
    )
    
    print("="*70)
    print(name)
    print("\nInput:")
    print(sentence)
    print("\nGenerated:")
    print(result)

# Step 27.3: Improving Generation Settings

During the first manual evaluation, the Kiswahili → Ekegusii direction showed a repetition problem.

To reduce this, we adjust the decoding strategy.

The following parameters are added:

- `repetition_penalty`
    - Discourages the model from repeatedly generating the same tokens.

- `no_repeat_ngram_size`
    - Prevents repeating the same sequence of words.

- `early_stopping`
    - Stops generation when the model reaches a complete output.

These changes only affect inference and do not modify the trained model.

In [ ]:
def translate_text(
    text,
    source_lang,
    target_lang,
    max_length=128
):
    
    # Set source language
    tokenizer.src_lang = source_lang
    
    
    # Tokenize input
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length
    )
    
    
    # Move tensors to GPU
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }
    
    
    # Target language ID
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(
        target_lang
    )
    
    
    # Generate translation
    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        
        # Improved decoding
        num_beams=5,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3,
        early_stopping=True,
        
        max_length=max_length
    )
    
    
    # Decode
    translation = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )[0]
    
    
    return translation


print("Updated translation function successfully.")

In [ ]:
swahili_sentence = (
    "Madarasa ya mahitaji maalum yameongezwa hadi 500 "
    "– yanaomba uhamisho kabla ya Desemba 20!"
)


result = translate_text(
    swahili_sentence,
    source_lang="swh_Latn",
    target_lang="guz_Latn"
)


print("Kiswahili:")
print(swahili_sentence)

print("\nGenerated Ekegusii:")
print(result)

# Step 28.1: Preparing Evaluation Metrics

Machine translation models are evaluated using automatic metrics.

We will use:

### BLEU
Measures the overlap between the generated translation and the reference translation.

### chrF
Measures character-level similarity between translations.

chrF is particularly useful for Ekegusii because Bantu languages often contain many morphological variations where character similarity gives more meaningful feedback than exact word matching.

In [ ]:
# Import evaluation libraries

import evaluate
import pandas as pd


# Load metrics

bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")


print("Evaluation metrics loaded successfully.")

# Step 28.2: Creating the Evaluation Function

A reusable evaluation function is created to measure translation performance.

For each translation direction, the function will:

1. Take the test examples.
2. Generate translations using the fine-tuned NLLB model.
3. Compare generated translations with the reference translations.
4. Calculate:
   - BLEU score
   - chrF score
5. Store sample translations for qualitative analysis.

The same evaluation procedure is applied to all four translation directions to ensure a fair comparison.

In [ ]:
from tqdm.auto import tqdm


def evaluate_translation_direction(
    dataset,
    source_lang,
    target_lang,
    num_samples=None
):
    
    predictions = []
    references = []
    examples = []
    
    
    # Limit samples if specified
    if num_samples:
        dataset = dataset.select(range(num_samples))
    
    
    for example in tqdm(dataset):
        
        source_text = example["source_text"]
        target_text = example["target_text"]
        
        
        # Generate translation
        prediction = translate_text(
            source_text,
            source_lang=source_lang,
            target_lang=target_lang
        )
        
        
        predictions.append(prediction)
        references.append([target_text])
        
        
        # Store examples
        examples.append({
            "source": source_text,
            "reference": target_text,
            "prediction": prediction
        })
    
    
    # Calculate BLEU
    bleu_score = bleu_metric.compute(
        predictions=predictions,
        references=references
    )
    
    
    # Calculate chrF
    chrf_score = chrf_metric.compute(
        predictions=predictions,
        references=references
    )
    
    
    return {
        "BLEU": bleu_score["score"],
        "chrF": chrf_score["score"],
        "examples": examples,
        "predictions": predictions,
        "references": references
    }


print("Evaluation function created successfully.")

# Step 28.3: Evaluating English → Ekegusii Translation

This step evaluates how well the fine-tuned NLLB model translates English public service announcements into Ekegusii.

The evaluation uses the test split only.

Metrics:

- BLEU: Measures similarity between generated translations and the reference Ekegusii translations.
- chrF: Measures character-level similarity, which is useful for low-resource languages such as Ekegusii.

The test is initially run on 100 examples to verify that the evaluation pipeline works correctly.

In [ ]:
# Evaluate English → Ekegusii

eng_guz_results = evaluate_translation_direction(
    test_dataset.filter(
        lambda x: x["source_lang"] == "eng_Latn"
        and x["target_lang"] == "guz_Latn"
    ),
    source_lang="eng_Latn",
    target_lang="guz_Latn",
    num_samples=100
)


print("English → Ekegusii")
print("--------------------")
print("BLEU score:", round(eng_guz_results["BLEU"], 2))
print("chrF score:", round(eng_guz_results["chrF"], 2))

# Step 28.3 Results: English → Ekegusii Evaluation

The fine-tuned NLLB model was evaluated on the English → Ekegusii translation direction using 100 unseen examples from the test dataset.

The evaluation metrics used were:

- **BLEU Score**
  - Measures the overlap between the generated translation and the reference translation.
  - Higher scores indicate that the generated translation contains more matching words and phrases compared to the expected translation.

- **chrF Score**
  - Measures character-level similarity between the generated translation and the reference translation.
  - This metric is especially useful for low-resource languages such as Ekegusii because morphological variations may cause different word forms while still preserving meaning.

## Results

| Translation Direction | BLEU | chrF |
|----------------------|------|------|
| English → Ekegusii | 5.07 | 26.59 |

## Interpretation

The English → Ekegusii direction achieved:

- **BLEU score: 5.07**
- **chrF score: 26.59**

The BLEU score is relatively low, which is expected for a newly introduced low-resource language in NLLB. Since Ekegusii is not originally supported by the pretrained NLLB model, the model must learn the language mapping mainly from the available fine-tuning dataset.

However, the chrF score of **26.59** indicates that the generated translations share a reasonable amount of character-level similarity with the reference Ekegusii translations. This suggests that although the model may not always produce exact word matches, it is learning Ekegusii word structures and morphological patterns.

The difference between BLEU and chrF indicates that the model is capturing some linguistic characteristics of Ekegusii but still struggles with generating exact translations.

Possible reasons for the lower BLEU score include:

1. **Limited training data**
   - The model was fine-tuned using approximately 5,000 Ekegusii examples, which is small compared to the multilingual data used during NLLB pretraining.

2. **New language integration**
   - Ekegusii (`guz_Latn`) was manually added to the tokenizer because it was not originally available in NLLB.
   - The model therefore does not have the same pretrained language representation as supported languages.

3. **Low-resource translation difficulty**
   - Translation into an underrepresented language is generally more challenging because fewer examples exist for learning vocabulary, grammar, and sentence structure.

Overall, the results show that the fine-tuned model is able to generate meaningful Ekegusii text, but additional data and training could further improve translation accuracy.

# Step 28.4: Qualitative Analysis of English → Ekegusii Results

Automatic evaluation metrics provide numerical measurements, but they do not always capture whether the generated translations preserve meaning.

Therefore, we inspect several examples from the English → Ekegusii evaluation.

For each example, we compare:

- Original English sentence
- Reference Ekegusii translation from the dataset
- Model-generated Ekegusii translation

This helps identify:
- Correctly translated concepts
- Vocabulary limitations
- Differences between generated and reference translations

In [ ]:
# Display sample English → Ekegusii translations

for i, example in enumerate(
    eng_guz_results["examples"][:10]
):
    
    print("=" * 80)
    print(f"Example {i+1}")
    
    print("\nEnglish:")
    print(example["source"])
    
    print("\nReference Ekegusii:")
    print(example["reference"])
    
    print("\nGenerated Ekegusii:")
    print(example["prediction"])

# Qualitative Analysis: English → Ekegusii Translation

The generated translations were manually inspected to understand the behavior of the fine-tuned NLLB model beyond numerical metrics.

The results show that the model has learned some translation patterns, but performance is inconsistent.

## Observations

### 1. Correct Ekegusii Generation in Some Cases

Some examples show that the model can generate meaningful Ekegusii translations.

For example:

**English:**
"Schools will remain open even if Covid-19 transmission remains high"

**Reference Ekegusii:**
"Chisukuru nchigenderere koigorwa nonya..."

The model preserved the meaning of keeping schools open and the Covid-19 context, although it produced the translation in another language.

This indicates that the model understands the semantic content of the input.

---

### 2. Language Control Problems

A common issue observed is that some outputs remain in English or Kiswahili instead of Ekegusii.

Examples:

**Input:**
"Free school bus service for 500,000 rural learners starts January!"

**Generated output:**
"free school bus service for 500,000 rural learners starts January!"

and:

**Input:**
"Use easy-opening windows in bedrooms for emergencies."

**Generated output:**
"windows ya kufungua kwa urahisi katika vyumba vya kulala wakati wa dharura."

The model generated the correct meaning, but not in the required target language.

This suggests that although the `guz_Latn` token was successfully introduced, the model still struggles with consistently generating Ekegusii because the language was not part of the original NLLB pretraining.

---

### 3. Semantic Understanding is Better Than Exact Translation

Some outputs preserve the meaning but use another language.

For example:

**Input:**
"Let's talk mental health! If they stopped talking, please check on them."

**Generated:**
"tuzungumze kuhusu afya ya akili! Kama wameacha kuzungumza, tafadhali angalia hali yao."

The translation is semantically correct, but it is Kiswahili rather than Ekegusii.

This explains the difference between the metrics:

- BLEU is low because exact Ekegusii word matching is missing.
- chrF is higher because the model still captures some linguistic patterns and sentence structure.

---

## Overall Assessment

The English → Ekegusii evaluation shows that the model has learned the relationship between English PSA content and the target domain, but it has difficulty enforcing Ekegusii generation consistently.

The main challenge is not understanding the input meaning, but producing the correct low-resource target language.

Possible improvements include:

1. Increasing the amount of Ekegusii training data.
2. Using additional Ekegusii monolingual data for continued pretraining.
3. Applying stronger language-control techniques during generation.
4. Training for more epochs with careful monitoring.

Overall, the model demonstrates initial capability for English → Ekegusii translation but requires further adaptation for reliable Ekegusii generation.

# Step 29: Evaluating Ekegusii → English Translation

This step evaluates the ability of the fine-tuned NLLB model to translate Ekegusii public service announcements into English.

Unlike English → Ekegusii, this direction tests whether the model can understand the newly introduced Ekegusii representation and map it into a high-resource language.

The evaluation uses:

- BLEU score
- chrF score

The test set is used because it contains unseen examples that were not used during training.

In [ ]:
# Evaluate Ekegusii → English

guz_eng_results = evaluate_translation_direction(
    test_dataset.filter(
        lambda x: x["source_lang"] == "guz_Latn"
        and x["target_lang"] == "eng_Latn"
    ),
    source_lang="guz_Latn",
    target_lang="eng_Latn",
    num_samples=100
)


print("Ekegusii → English")
print("--------------------")
print("BLEU score:", round(guz_eng_results["BLEU"], 2))
print("chrF score:", round(guz_eng_results["chrF"], 2))

# Step 29 Results: Ekegusii → English Evaluation

The fine-tuned NLLB model was evaluated on the Ekegusii → English translation direction using 100 unseen examples from the test dataset.

## Results

| Translation Direction | BLEU | chrF |
|----------------------|------|------|
| Ekegusii → English | 8.43 | 33.11 |

## Interpretation

The Ekegusii → English direction achieved:

- **BLEU score: 8.43**
- **chrF score: 33.11**

Compared with English → Ekegusii, this direction achieved higher scores:

| Direction | BLEU | chrF |
|-|-|-|
| English → Ekegusii | 5.07 | 26.59 |
| Ekegusii → English | 8.43 | 33.11 |

This indicates that the model performs better when translating from Ekegusii into a high-resource language (English) than when generating Ekegusii output.

## Analysis

The improved performance can be explained by several factors:

### 1. Strong English Representation

English was already included in the original NLLB pretraining data. Therefore, the model has strong knowledge of English vocabulary, grammar, and sentence generation.

When translating into English, the model can rely on existing language representations.

### 2. Difficulty Generating Ekegusii

For English → Ekegusii translation, the model must generate a language that was not originally supported by NLLB.

Although the `guz_Latn` token was added and the model was fine-tuned, the amount of Ekegusii training data is relatively small compared to the languages available during pretraining.

### 3. chrF Improvement

The chrF score of **33.11** indicates that the model captures a reasonable amount of character-level similarity with the reference translations.

This suggests that the model has learned Ekegusii word patterns and can interpret Ekegusii input, even though exact translation matching remains challenging.

## Conclusion

The results show that the model has developed the ability to understand Ekegusii text and translate it into English. However, generating Ekegusii remains the more challenging task, highlighting the difficulty of extending multilingual models to unsupported low-resource languages.

# Step 29.1: Qualitative Analysis of Ekegusii → English

To better understand model performance, several Ekegusii → English translations are inspected manually.

Each example compares:

- Original Ekegusii input
- Reference English translation
- Generated English translation

This analysis helps identify whether the model preserves meaning and produces fluent English output.

In [ ]:
# Display Ekegusii → English examples

for i, example in enumerate(
    guz_eng_results["examples"][:10]
):
    
    print("=" * 80)
    print(f"Example {i+1}")
    
    print("\nEkegusii:")
    print(example["source"])
    
    print("\nReference English:")
    print(example["reference"])
    
    print("\nGenerated English:")
    print(example["prediction"])

# Qualitative Analysis: Ekegusii → English Translation

To complement the automatic evaluation metrics, several Ekegusii → English translations were manually inspected.

The analysis compares the original Ekegusii sentence, the reference English translation, and the generated English output.

## Observations

### 1. The Model Successfully Captures General Meaning

Several examples show that the model understands the overall topic of the input.

Example:

**Ekegusii Input:**
CBC lessons on Radio Taifa from 10 AM daily.

**Generated Output:**
Courses on CBC are available via Radio Taifa starting at 4pm daily.

The model correctly identified:
- CBC education content
- Radio Taifa
- Daily scheduling information

Although the exact time was incorrect, the main concept was preserved.

---

### 2. Good Fluency in English Generation

Compared with English → Ekegusii, the generated outputs are generally fluent and grammatically correct English.

For example:

**Generated:**
"An online platform has been launched to highlight protests across Kenya, including police deployments, and a digital platform to support protesters."

The output is coherent and follows normal English sentence structure.

This demonstrates that the model benefits from the strong English representation learned during NLLB pretraining.

---

### 3. Semantic Drift and Information Loss

A major limitation is that some translations lose important details or introduce incorrect information.

Example:

Reference:

"All boarding schools must install fire alarms + sprinklers by Jan 6 — CS orders!"

Generated:

"All primary schools must register for water supplies in case of floods."

The model recognizes the education and safety context but changes important concepts:

- boarding schools → primary schools
- fire alarms/sprinklers → water supplies/floods

This indicates incomplete understanding of specific Ekegusii vocabulary.

---

### 4. Difficulty with Domain-Specific Vocabulary

Some PSA topics contain specialized terms:

- government policies
- agriculture
- public health
- climate information
- security announcements

The model sometimes produces related but incorrect terms.

Example:

Reference:

"Taxation and policy biases against agriculture..."

Generated:

"Legislation and regulations to protect farmers..."

The model identifies the agriculture policy domain but does not preserve the exact message.

---

## Overall Assessment

The Ekegusii → English results show stronger performance than English → Ekegusii.

The model demonstrates:

### Strengths:
- Good English generation ability.
- Ability to identify PSA topics.
- Partial preservation of meaning.
- Understanding of common Ekegusii structures.

### Limitations:
- Inaccurate translation of specific details.
- Hallucinated information in some cases.
- Difficulty with specialized terminology.
- Limited Ekegusii representation due to low-resource training.

Overall, the results indicate that fine-tuning successfully improved the model's ability to process Ekegusii text, but more data and further adaptation would be required for high-accuracy translation.

# Step 30: Evaluating Kiswahili → Ekegusii Translation

This step evaluates the performance of the fine-tuned NLLB model when translating Kiswahili public service announcements into Ekegusii.

Kiswahili and Ekegusii are both Bantu languages, which may provide some linguistic advantages compared with English → Ekegusii translation.

The evaluation uses:

- BLEU score
- chrF score

The test set contains unseen Kiswahili → Ekegusii examples.

In [ ]:
# Evaluate Kiswahili → Ekegusii

swh_guz_results = evaluate_translation_direction(
    test_dataset.filter(
        lambda x: x["source_lang"] == "swh_Latn"
        and x["target_lang"] == "guz_Latn"
    ),
    source_lang="swh_Latn",
    target_lang="guz_Latn",
    num_samples=100
)


print("Kiswahili → Ekegusii")
print("--------------------")
print("BLEU score:", round(swh_guz_results["BLEU"], 2))
print("chrF score:", round(swh_guz_results["chrF"], 2))

# Step 30 Results: Kiswahili → Ekegusii Evaluation

The fine-tuned NLLB model was evaluated on the Kiswahili → Ekegusii translation direction using 100 unseen examples from the test dataset.

## Results

| Translation Direction | BLEU | chrF |
|----------------------|------|------|
| Kiswahili → Ekegusii | 2.61 | 20.91 |

## Interpretation

The Kiswahili → Ekegusii direction achieved:

- **BLEU score: 2.61**
- **chrF score: 20.91**

This is currently the lowest-performing translation direction.

Although Kiswahili and Ekegusii are both Bantu languages, the model still struggles to generate Ekegusii accurately.

## Analysis

### 1. Shared Language Family Does Not Guarantee Strong Translation

Kiswahili and Ekegusii share some Bantu linguistic characteristics, including:

- Similar noun class systems.
- Agglutinative word formation.
- Related grammatical structures.

However, the languages are still significantly different in vocabulary and sentence construction.

Therefore, being in the same language family provides some linguistic similarity but does not replace the need for sufficient training data.

---

### 2. Target-Side Ekegusii Generation Remains the Main Challenge

Similar to English → Ekegusii, the model has difficulty producing Ekegusii output.

This suggests that the main limitation is not only the source language, but the target language representation.

The pattern observed is:

| Direction | Performance |
|-|-|
| Translation into Ekegusii | Lower performance |
| Translation from Ekegusii | Higher performance |

This indicates that the model has learned some Ekegusii understanding, but Ekegusii generation remains weak.

---

### 3. Limited Ekegusii Data

The model was fine-tuned using approximately 5,126 Ekegusii parallel examples.

For a newly introduced language token, this amount of data is relatively small.

The model needs more examples to learn:

- Ekegusii vocabulary.
- Word endings.
- Sentence patterns.
- Domain-specific PSA terminology.

---

## Conclusion

The Kiswahili → Ekegusii results show that linguistic similarity between languages can provide some benefit, but it is not sufficient for introducing a new low-resource language into a multilingual model.

The main challenge remains building a strong Ekegusii generation capability.

# Step 31: Evaluating Ekegusii → Kiswahili Translation

This step evaluates the fine-tuned NLLB model when translating Ekegusii public service announcements into Kiswahili.

Kiswahili is already supported by the original NLLB model, unlike Ekegusii.

Therefore, this direction evaluates whether the model can combine:

- The newly learned Ekegusii representation.
- The existing Kiswahili language capability.

Evaluation metrics:

- BLEU score
- chrF score

In [ ]:
# Evaluate Ekegusii → Kiswahili

guz_swh_results = evaluate_translation_direction(
    test_dataset.filter(
        lambda x: x["source_lang"] == "guz_Latn"
        and x["target_lang"] == "swh_Latn"
    ),
    source_lang="guz_Latn",
    target_lang="swh_Latn",
    num_samples=100
)


print("Ekegusii → Kiswahili")
print("--------------------")
print("BLEU score:", round(guz_swh_results["BLEU"], 2))
print("chrF score:", round(guz_swh_results["chrF"], 2))

# Overall Evaluation Results

The fine-tuned NLLB model was evaluated across all four translation directions using unseen examples from the test dataset.

## Overall Results

| Translation Direction | BLEU | chrF |
|------------------------|------:|------:|
| English → Ekegusii | 5.07 | 26.59 |
| Ekegusii → English | 8.43 | 33.11 |
| Kiswahili → Ekegusii | 2.61 | 20.91 |
| Ekegusii → Kiswahili | 9.39 | 37.42 |

## Discussion

The evaluation demonstrates a clear difference between translation into Ekegusii and translation from Ekegusii.

The highest performance was achieved for **Ekegusii → Kiswahili**, followed by **Ekegusii → English**. Both English and Kiswahili are high-resource languages that were already included during NLLB pretraining. Consequently, the model was able to generate fluent output in these languages after learning to interpret Ekegusii during fine-tuning.

In contrast, the two translation directions targeting Ekegusii achieved lower BLEU and chrF scores. This indicates that generating fluent Ekegusii remains significantly more difficult than understanding it.

One possible reason is that Ekegusii (`guz_Latn`) was not originally supported by NLLB. Although a new language token was introduced and the model was fine-tuned using approximately 5,126 parallel sentence pairs, the amount of training data is relatively small compared with the multilingual corpus used during pretraining.

Interestingly, **Kiswahili → Ekegusii** produced the lowest scores despite Kiswahili and Ekegusii both belonging to the Bantu language family. This suggests that linguistic similarity alone is insufficient for accurate translation when the target language lacks strong pretrained representations. Adequate parallel data remains essential for learning vocabulary, grammar, and sentence structure.

Overall, the results indicate that the proposed approach successfully enabled bidirectional translation involving Ekegusii. The model demonstrated a stronger ability to understand Ekegusii than to generate it, highlighting both the feasibility and the challenges of extending multilingual neural machine translation models to previously unsupported low-resource African languages.

# Step 32: Final Evaluation on the Complete Test Set

The previous evaluation used 100 randomly selected test examples to verify that the evaluation pipeline was functioning correctly.

To obtain the final performance of the fine-tuned model, evaluation is now performed on the entire test dataset. Evaluating on all unseen test examples provides more reliable BLEU and chrF scores and ensures that the reported results are representative of the model's overall translation performance.

The same evaluation procedure is applied to all four translation directions:

- English → Ekegusii
- Ekegusii → English
- Kiswahili → Ekegusii
- Ekegusii → Kiswahili

In [ ]:
translation_directions = [
    ("eng_Latn", "guz_Latn", "English → Ekegusii"),
    ("guz_Latn", "eng_Latn", "Ekegusii → English"),
    ("swh_Latn", "guz_Latn", "Kiswahili → Ekegusii"),
    ("guz_Latn", "swh_Latn", "Ekegusii → Kiswahili"),
]

final_results = []

for source_lang, target_lang, direction in translation_directions:

    print("=" * 70)
    print(direction)

    subset = test_dataset.filter(
        lambda x: x["source_lang"] == source_lang and x["target_lang"] == target_lang
    )

    results = evaluate_translation_direction(
        subset,
        source_lang=source_lang,
        target_lang=target_lang
    )

    final_results.append({
        "Direction": direction,
        "BLEU": results["BLEU"],
        "chrF": results["chrF"]
    })

    print("BLEU :", round(results["BLEU"], 2))
    print("chrF :", round(results["chrF"], 2))

# Final Evaluation Results

The fine-tuned NLLB model was evaluated on the complete test dataset across all four translation directions using BLEU and chrF metrics.

## Overall Performance

| Translation Direction | BLEU | chrF |
|------------------------|------:|------:|
| English → Ekegusii | 3.70 | 26.45 |
| Ekegusii → English | 9.10 | 33.13 |
| Kiswahili → Ekegusii | 2.83 | 22.82 |
| Ekegusii → Kiswahili | 11.00 | 38.03 |

## Discussion

The evaluation demonstrates that the proposed fine-tuned NLLB model is capable of translating between English, Kiswahili, and Ekegusii in both directions. However, translation performance differs considerably depending on the target language.

The highest performance was achieved for **Ekegusii → Kiswahili** (BLEU = 11.00, chrF = 38.03), followed by **Ekegusii → English** (BLEU = 9.10, chrF = 33.13). These results indicate that the model is better at interpreting Ekegusii and generating translations in languages that were already well represented during NLLB pretraining.

Conversely, the two translation directions targeting Ekegusii achieved lower BLEU and chrF scores. English → Ekegusii obtained a BLEU score of 3.70 and a chrF score of 26.45, while Kiswahili → Ekegusii achieved the lowest BLEU score of 2.83 and a chrF score of 22.82.

These findings suggest that the principal challenge lies in generating fluent Ekegusii rather than understanding it. Since Ekegusii was not originally supported by NLLB, the model relied entirely on the added language token and the available fine-tuning data to learn its vocabulary and grammatical structure. Although the model successfully generated meaningful Ekegusii translations in many cases, generation remained less consistent than for the pretrained target languages.

Interestingly, despite Kiswahili and Ekegusii both belonging to the Bantu language family, Kiswahili → Ekegusii produced the lowest translation accuracy. This suggests that linguistic relatedness alone is insufficient for high-quality translation when the target language lacks extensive pretrained representations and large parallel corpora.

Overall, the results demonstrate that the proposed approach successfully extends NLLB to support bidirectional translation involving Ekegusii. While translation into Ekegusii remains challenging, the model shows promising capability in understanding Ekegusii and translating it into high-resource languages, providing a strong foundation for future improvements using larger datasets and additional adaptation techniques.

## Limitations

This study has several limitations:

- Ekegusii was not part of the original NLLB pretrained language set and therefore required manual integration through the addition of a new language token.
- The fine-tuning dataset consisted of approximately 5,126 parallel sentence pairs, which is relatively small for training a neural machine translation model.
- The dataset was domain-specific, focusing on public service announcements, which may limit the model's ability to generalize to other text domains.
- Due to computational constraints, the model was fine-tuned for three epochs. Additional training or larger datasets may further improve translation quality.

# Step 34: Improving Model Performance Through Target-Language Oversampling

The baseline evaluation showed that the model performs better when translating **from Ekegusii** into other languages than when translating **into Ekegusii**.

The results indicated that Ekegusii generation is the main challenge:

| Translation Direction | BLEU Score |
|----------------------|-----------:|
| English → Ekegusii | 3.70 |
| Kiswahili → Ekegusii | 2.83 |
| Ekegusii → English | 9.10 |
| Ekegusii → Kiswahili | 11.00 |

The lower scores for English → Ekegusii and Kiswahili → Ekegusii suggest that the model requires more exposure to examples where Ekegusii is the target language.

## Objective

To improve Ekegusii generation, the training dataset will be modified using **target-language oversampling**.

Oversampling increases the number of training examples for weaker translation directions without introducing new data. This allows the model to receive more optimization updates focused on learning Ekegusii vocabulary, sentence structure, and grammatical patterns.

## Approach

The original training dataset contains four translation directions with approximately equal representation:

- English → Ekegusii
- Ekegusii → English
- Kiswahili → Ekegusii
- Ekegusii → Kiswahili

For the improved experiment:

- English → Ekegusii examples will be duplicated.
- Kiswahili → Ekegusii examples will be duplicated.
- Ekegusii → English examples will remain unchanged.
- Ekegusii → Kiswahili examples will remain unchanged.

This creates a training distribution that places more emphasis on generating Ekegusii while preserving the model's ability to translate from Ekegusii.

## Expected Impact

By increasing the frequency of Ekegusii-target examples during training, the model may improve its ability to:

- Generate more accurate Ekegusii words.
- Learn Ekegusii sentence patterns.
- Reduce copying of source-language text.
- Reduce language switching errors.

The improved model will later be evaluated using the same BLEU and chrF metrics and compared against the baseline model.

In [20]:
baseline_path = "nllb-guz-baseline"
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(baseline_path, local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(baseline_path, local_files_only=True)
print("guz_Latn check:", len(tokenizer), model.get_input_embeddings().num_embeddings)

Loading weights: 100%|██████████| 509/509 [00:00<00:00, 3787.34it/s]


guz_Latn check: 256205 256205


In [21]:
eng_to_eke = pd.DataFrame({"source_text": df["English_clean"], "target_text": df["Ekegusii_clean"], "source_lang": "eng_Latn", "target_lang": "guz_Latn"})
eke_to_eng = pd.DataFrame({"source_text": df["Ekegusii_clean"], "target_text": df["English_clean"], "source_lang": "guz_Latn", "target_lang": "eng_Latn"})
swh_to_eke = pd.DataFrame({"source_text": df["Kiswahili_clean"], "target_text": df["Ekegusii_clean"], "source_lang": "swh_Latn", "target_lang": "guz_Latn"})
eke_to_swh = pd.DataFrame({"source_text": df["Ekegusii_clean"], "target_text": df["Kiswahili_clean"], "source_lang": "guz_Latn", "target_lang": "swh_Latn"})

translation_df = pd.concat([eng_to_eke, eke_to_eng, swh_to_eke, eke_to_swh], ignore_index=True)
translation_df = translation_df.drop_duplicates(subset=["source_text", "target_text"]).reset_index(drop=True)

train_df, temp_df = train_test_split(translation_df, test_size=0.2, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)

from datasets import Dataset, DatasetDict, concatenate_datasets
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))
dataset = DatasetDict({"train": train_dataset, "validation": val_dataset, "test": test_dataset})

train_data = dataset["train"]
eng_to_guz = train_data.filter(lambda x: x["source_lang"] == "eng_Latn" and x["target_lang"] == "guz_Latn")
swh_to_guz = train_data.filter(lambda x: x["source_lang"] == "swh_Latn" and x["target_lang"] == "guz_Latn")
oversampled_train = concatenate_datasets([train_data, eng_to_guz, swh_to_guz])
print("Oversampled train size:", len(oversampled_train))

Filter: 100%|██████████| 16188/16188 [00:00<00:00, 196188.18 examples/s]

Oversampled train size: 24323


# Step 35: Verifying the Improved Training Distribution

Before retraining the model, we verify that oversampling produced the intended training distribution.

The goal is to confirm that translation directions where Ekegusii is the target language now appear more frequently.

This ensures that the model receives more training updates focused on Ekegusii generation while maintaining examples for translation from Ekegusii.

In [22]:
from collections import Counter

distribution = Counter(
    [
        (x["source_lang"], x["target_lang"])
        for x in oversampled_train
    ]
)

for pair, count in distribution.items():
    print(pair, ":", count)

('swh_Latn', 'guz_Latn') : 8198
('guz_Latn', 'eng_Latn') : 3941
('eng_Latn', 'guz_Latn') : 8072
('guz_Latn', 'swh_Latn') : 4112


# Step 36: Fine-Tuning the Improved NLLB Model

In this experiment, the baseline model is further fine-tuned using the oversampled training dataset.

The previous evaluation showed that the model was stronger at understanding Ekegusii than generating it. Therefore, the improved training strategy increases the frequency of examples where Ekegusii is the target language.

## Training Changes Compared With Baseline

| Component | Baseline | Improved Model |
|-----------|----------|----------------|
| Training data | 16,188 examples | 24,323 examples |
| Sampling strategy | Equal distribution | Ekegusii-target oversampling |
| Epochs | 3 | 10 |
| Starting model | NLLB fine-tuned model | Baseline fine-tuned model |

## Objective

The objective of this experiment is to improve:

- English → Ekegusii translation quality.
- Kiswahili → Ekegusii translation quality.
- Ekegusii vocabulary generation.
- Fluency and grammatical consistency.

The improved model will later be evaluated using the same BLEU and chrF metrics to measure whether oversampling improved performance compared with the baseline.

# Step 37: Loading the Baseline Model

The improved experiment starts from the previously trained baseline model instead of the original NLLB checkpoint.

This approach allows the model to retain the knowledge learned during the first fine-tuning stage while receiving additional training focused on improving Ekegusii generation.

The baseline model contains:

- The adapted Ekegusii language token (`guz_Latn`).
- Learned Ekegusii vocabulary patterns.
- Previous translation knowledge.

The model will now be further fine-tuned using the oversampled training dataset.

# Step 38: Tokenizing the Oversampled Training Dataset

Transformer models cannot process raw text directly. The input sentences must first be converted into token IDs using the NLLB tokenizer.

In this step:

- The oversampled training dataset is tokenized.
- The validation dataset is tokenized using the same process.
- Source language and target language codes are used to ensure the model knows the translation direction.

The tokenizer converts:

Raw text:

"Monitor nutrition in school feeding programs 2020."

into:

- Input token IDs
- Attention masks
- Target labels

These numerical representations are then used during fine-tuning.

The validation dataset is kept unchanged because it is used to measure whether improvements generalize to unseen data.

In [23]:
def tokenize_batch(examples):
    input_ids_list, attention_mask_list, labels_list = [], [], []
    for src_text, tgt_text, src_lang, tgt_lang in zip(
        examples["source_text"], examples["target_text"],
        examples["source_lang"], examples["target_lang"]
    ):
        tokenizer.src_lang = src_lang
        enc = tokenizer(src_text, max_length=128, truncation=True)
        input_ids_list.append(enc["input_ids"])
        attention_mask_list.append(enc["attention_mask"])

        tgt_id = tokenizer.convert_tokens_to_ids(tgt_lang)
        tgt_ids = tokenizer(tgt_text, max_length=127, truncation=True, add_special_tokens=False)["input_ids"]
        labels_list.append([tgt_id] + tgt_ids + [tokenizer.eos_token_id])
    return {"input_ids": input_ids_list, "attention_mask": attention_mask_list, "labels": labels_list}

tokenized_train = oversampled_train.map(tokenize_batch, batched=True, batch_size=16, remove_columns=oversampled_train.column_names)
tokenized_validation = dataset["validation"].map(tokenize_batch, batched=True, batch_size=16, remove_columns=dataset["validation"].column_names)

# Confirm the fix -- first token should now be guz_Latn/eng_Latn/swh_Latn, not a word-piece
print(tokenizer.convert_ids_to_tokens(tokenized_train[0]["labels"][:1]))

Map: 100%|██████████| 2023/2023 [00:00<00:00, 3260.23 examples/s]

['guz_Latn']


# Step 39: Configuring the Improved Model Training Setup

The improved model will be trained using the oversampled dataset created in the previous step.

The training configuration is adjusted from the baseline experiment to improve resource management and training efficiency.

## Training Configuration

The improved model will use:

- **Starting model:** Previously saved baseline model.
- **Training dataset:** Oversampled dataset (24,323 examples).
- **Training epochs:** 10 epochs.
- **GPU acceleration:** Enabled using mixed precision training.
- **Checkpoint saving:** Limited to one checkpoint to prevent storage exhaustion.
- **Evaluation:** Performed after each epoch.

## Storage Optimization

During the first training experiment, multiple checkpoints caused the Kaggle storage limit to be reached.

To avoid this:

- Only the most recent checkpoint will be retained.
- Optimizer states from unnecessary checkpoints will not accumulate.
- The final trained model will be saved separately after training.

The resulting model will be stored as:nllb-ekegusii-improved and will be compared against baseline model


In [24]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

output_dir = "./nllb-ekegusii-improved-v2"

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=10,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    predict_with_generate=True,
    fp16=True,
    logging_steps=100,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    processing_class=tokenizer,
    data_collator=data_collator,
)
print("Trainer ready")

Trainer ready


# Step 40: Fine-Tuning the Improved NLLB Model

The improved model is trained using the oversampled dataset created earlier.

Unlike the baseline experiment, this training run gives more importance to translation tasks where Ekegusii is the target language.

The model starts from the baseline checkpoint rather than the original NLLB model. This allows it to retain previously learned translation patterns while adapting further to improve Ekegusii generation.

## Training Objective

The objective is to improve:

- English → Ekegusii translation quality.
- Kiswahili → Ekegusii translation quality.
- Ekegusii vocabulary generation.
- Sentence fluency and grammatical consistency.

## Training Settings

The improved training uses:

- Epochs: 5
- Learning rate: 2e-5
- Batch size: 4
- GPU acceleration: Enabled
- Mixed precision training: Enabled
- Checkpoints saved: Maximum 1

After training, the model will be saved and evaluated using BLEU and chrF scores, then compared against the baseline model.

In [26]:
import torch
import shutil
import os

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Check free storage on Navon's local disk
total, used, free = shutil.disk_usage("/home/jovyan")
print(
    "Free storage:",
    round(free/(1024**3), 2),
    "GB"
)

# Check the actual output path used in training_args (local folder now)
print(
    "Output folder exists:",
    os.path.exists("./nllb-ekegusii-improved")
)

GPU available: True
GPU: NVIDIA A100-SXM4-80GB
Free storage: 694.36 GB
Output folder exists: False


In [27]:
training_results = trainer.train()

Epoch,Training Loss,Validation Loss
1,2.672827,2.436689
2,2.381135,2.235422
3,2.154447,2.107356
4,2.025520,2.017377
5,1.922247,1.956047
6,1.837328,1.908933
7,1.756971,1.872620
8,1.721315,1.850953
9,1.682256,1.836757
10,1.682253,1.833234


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.84s/it]
[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


In [28]:
#Sanity check
guz_id = tokenizer.convert_tokens_to_ids("guz_Latn")
model.generation_config.forced_bos_token_id = guz_id

tokenizer.src_lang = "eng_Latn"
inputs = tokenizer("Use easy-opening windows in bedrooms for emergencies.", return_tensors="pt").to(model.device)
out = model.generate(**inputs, forced_bos_token_id=guz_id, max_length=128, num_beams=4)
print(tokenizer.decode(out[0], skip_special_tokens=True))

Kabe nomoroberio bwokobeka emerangu ase chinyomba chiokorara ekero kiemebasekano.


# Step 41: Evaluating the Improved NLLB Model

After completing 10 epochs of fine-tuning on the oversampled dataset, the improved model is evaluated using the same procedure applied to the baseline model.

The evaluation uses:

- **BLEU score** — measures n-gram overlap with the reference translation.
- **chrF score** — measures character-level similarity, useful for capturing partial correctness in a low-resource language like Ekegusii.

The same four translation directions are evaluated on the full test set:

1. English → Ekegusii
2. Ekegusii → English
3. Kiswahili → Ekegusii
4. Ekegusii → Kiswahili

This allows a direct, like-for-like comparison against the baseline model's results.

In [29]:
import evaluate
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def evaluate_translation_direction(direction_name, src_lang, tgt_lang, test_subset, max_samples=None):
    print(f"\n{'='*70}")
    print(direction_name)
    print('='*70)

    subset = test_subset if max_samples is None else test_subset.select(range(min(max_samples, len(test_subset))))
    tokenizer.src_lang = src_lang
    forced_id = tokenizer.convert_tokens_to_ids(tgt_lang)

    model.eval()
    predictions, references = [], []

    from tqdm import tqdm
    for row in tqdm(subset):
        inputs = tokenizer(row["source_text"], return_tensors="pt", truncation=True, max_length=128).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, forced_bos_token_id=forced_id, max_length=128, num_beams=4)
        predictions.append(tokenizer.decode(out[0], skip_special_tokens=True))
        references.append(row["target_text"])

    bleu = bleu_metric.compute(predictions=predictions, references=[[r] for r in references])
    chrf = chrf_metric.compute(predictions=predictions, references=[[r] for r in references])
    print(f"BLEU : {bleu['score']:.2f}")
    print(f"chrF : {chrf['score']:.2f}")
    return bleu['score'], chrf['score']

eng_to_guz_test = test_dataset.filter(lambda x: x["source_lang"] == "eng_Latn" and x["target_lang"] == "guz_Latn")
guz_to_eng_test = test_dataset.filter(lambda x: x["source_lang"] == "guz_Latn" and x["target_lang"] == "eng_Latn")
swh_to_guz_test = test_dataset.filter(lambda x: x["source_lang"] == "swh_Latn" and x["target_lang"] == "guz_Latn")
guz_to_swh_test = test_dataset.filter(lambda x: x["source_lang"] == "guz_Latn" and x["target_lang"] == "swh_Latn")

results = {}
results["eng_to_guz"] = evaluate_translation_direction("English → Ekegusii", "eng_Latn", "guz_Latn", eng_to_guz_test)
results["guz_to_eng"] = evaluate_translation_direction("Ekegusii → English", "guz_Latn", "eng_Latn", guz_to_eng_test)
results["swh_to_guz"] = evaluate_translation_direction("Kiswahili → Ekegusii", "swh_Latn", "guz_Latn", swh_to_guz_test)
results["guz_to_swh"] = evaluate_translation_direction("Ekegusii → Kiswahili", "guz_Latn", "swh_Latn", guz_to_swh_test)

Filter: 100%|██████████| 2024/2024 [00:00<00:00, 170439.92 examples/s]



English → Ekegusii


100%|██████████| 500/500 [06:44<00:00,  1.23it/s]


BLEU : 17.04
chrF : 47.71

Ekegusii → English


100%|██████████| 525/525 [04:06<00:00,  2.13it/s]


BLEU : 16.89
chrF : 41.71

Kiswahili → Ekegusii


100%|██████████| 504/504 [08:11<00:00,  1.03it/s]


BLEU : 15.83
chrF : 48.12

Ekegusii → Kiswahili


100%|██████████| 495/495 [05:01<00:00,  1.64it/s]


BLEU : 17.18
chrF : 44.37


## Comparison: Baseline vs. Oversampled Model 

| Direction | Baseline BLEU | Oversampled BLEU (corrected) | Δ BLEU | Baseline chrF | Oversampled chrF (corrected) | Δ chrF |
|---|---:|---:|---:|---:|---:|---:|
| English → Ekegusii | 3.70 | 17.04 | **+13.34** | 26.45 | 47.71 | **+21.26** |
| Ekegusii → English | 9.10 | 16.89 | +7.79 | 33.13 | 41.71 | +8.58 |
| Kiswahili → Ekegusii | 2.83 | 15.83 | **+13.00** | 22.82 | 48.12 | **+25.30** |
| Ekegusii → Kiswahili | 11.00 | 17.18 | +6.18 | 38.03 | 44.37 | +6.34 |

### Interpretation

With the labeling bug fixed, the oversampling strategy's benefit is even 
clearer than the earlier (bugged) numbers suggested. The two 
Ekegusii-as-target directions — the ones deliberately oversampled — again 
show the largest gains by a wide margin:

- **English→Ekegusii**: +13.34 BLEU / +21.26 chrF — the largest chrF gain of 
  any direction.
- **Kiswahili→Ekegusii**: +13.00 BLEU / +25.30 chrF — now the single largest 
  improvement in the entire table, in both metrics.

**The original asymmetry has nearly closed.** At baseline, the gap between 
the strongest and weakest direction was 8.17 BLEU (Ekegusii→Kiswahili at 
11.00 vs. Kiswahili→Ekegusii at 2.83). After the corrected oversampling run, 
that gap has shrunk to just **1.35 BLEU** (Ekegusii→Kiswahili at 17.18 vs. 
Kiswahili→Ekegusii at 15.83) — all four directions now sit within a narrow 
15.8-17.2 BLEU band. On chrF, Kiswahili→Ekegusii (48.12) is now actually the 
**strongest** direction, not the weakest — a full reversal from baseline.



In [31]:
import torch
def show_translations(direction_name, src_lang, tgt_lang, test_subset, n=5):
    print(f"\n{'='*70}")
    print(direction_name)
    print('='*70)
    tokenizer.src_lang = src_lang
    forced_id = tokenizer.convert_tokens_to_ids(tgt_lang)
    model.eval()
    sample = test_subset.select(range(min(n, len(test_subset))))
    for row in sample:
        inputs = tokenizer(row["source_text"], return_tensors="pt", truncation=True, max_length=128).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, forced_bos_token_id=forced_id, max_length=128, num_beams=4)
        prediction = tokenizer.decode(out[0], skip_special_tokens=True)
        print(f"\nSource:    {row['source_text']}")
        print(f"Predicted: {prediction}")
        print(f"Reference: {row['target_text']}")
# Filter the test set by direction (reuse the same filtering pattern from
# the evaluation cells) and show 5 examples from each
eng_to_guz_test = test_dataset.filter(lambda x: x["source_lang"] == "eng_Latn" and x["target_lang"] == "guz_Latn")
guz_to_eng_test = test_dataset.filter(lambda x: x["source_lang"] == "guz_Latn" and x["target_lang"] == "eng_Latn")
swh_to_guz_test = test_dataset.filter(lambda x: x["source_lang"] == "swh_Latn" and x["target_lang"] == "guz_Latn")
guz_to_swh_test = test_dataset.filter(lambda x: x["source_lang"] == "guz_Latn" and x["target_lang"] == "swh_Latn")
show_translations("English → Ekegusii", "eng_Latn", "guz_Latn", eng_to_guz_test)
show_translations("Ekegusii → English", "guz_Latn", "eng_Latn", guz_to_eng_test)
show_translations("Kiswahili → Ekegusii", "swh_Latn", "guz_Latn", swh_to_guz_test)
show_translations("Ekegusii → Kiswahili", "guz_Latn", "swh_Latn", guz_to_swh_test)

Filter: 100%|██████████| 2024/2024 [00:00<00:00, 159010.85 examples/s]



English → Ekegusii

Source:    English_text
Predicted: English_text
Reference: Egetabu kiegesongo

Source:    Use easy-opening windows in bedrooms for emergencies.
Predicted: Kabe nomoroberio bwokobeka emerangu ase chinyomba chiokorara ekero kiemebasekano.
Reference: Beka chitirisa chikoigoka kebwororo ase chinyomba chiokorara koba aororo ekero kiemebasekano kominchoria.

Source:    Free school bus service for 500,000 rural learners starts January!
Predicted: Chibesa chia bosa ase abana 500,000 baria bagosoma chisukuru chia risabu ngochaka omotienyi omotang'ani!
Reference: Chiroti chia bosa ase abana baria bagoma ime yechinsinyo chiense 500,000 ngochaka ore omotienyi omotang'ani!

Source:    Learn one way of health in appreciating human health, animal and public heàlth and as a means of reducing animal diseases.
Predicted: Beegie enchera eyemo yobochenu ase okomanya obochenu bwa mwanyabanto, etugo amo nobochenu bwabanto na buna enchera eyemo yogokei amarwaire yetugo.
Reference: Beegie

## Interpretation: Qualitative Review of the Corrected Model

### 1. The target-language-confusion bug is gone
Every single one of the 20 examples now generates output in the correct 
target language — no more Kiswahili appearing where Ekegusii was requested, 
or English where Ekegusii was requested. This directly confirms the label 
fix worked, not just at the metric level but at the level of actual usable 
output. Compare directly to the earlier buggy run's "Use easy-opening 
windows..." example, which produced Kiswahili — the corrected model now 
produces genuine Ekegusii for the exact same sentence: *"Kabe nomoroberio 
bwokobeka emerangu ase chinyomba chiokorara ekero kiemebasekano"* — sharing 
real vocabulary with the reference (`chinyomba`, `kiemebasekano`).


### 2. Semantic accuracy is noticeably improved, not just fluency
Several examples that showed genuine factual drift in the buggy run are now 
much closer to the reference:
- The flood-safety PSA now correctly conveys "learn to adapt to climate 
  change, store supplies, avoid flood areas" — closely matching the 
  reference's intent, versus the earlier run's reasonable-but-less-precise 
  paraphrase.
- The GenZ protest/COP policing example is now an almost exact match to the 
  reference's meaning, just with minor word-choice differences.
- The Mama Mboga/COVID-19 example still shows some factual drift 
  ("food from home" vs. reference's "cutting vegetables for customers"), so 
  this specific failure mode — losing precise details on some sentences — 
  persists even after the fix, worth keeping in the limitations section 
  rather than claiming it's fully resolved.

### 3. One new, minor artifact worth noting: repetition looping
The Kiswahili→Ekegusii malaria/climate example shows a small repetition 
glitch — *"Ense riria riria riria rikirorekanete"* repeats "riria" three 
times, a known failure mode in beam-search generation on longer, more 
complex source sentences. This is a new observation from this batch, not 
seen as clearly in the earlier review, and worth adding as a specific, 
concrete limitation (beam search occasionally loops on long/complex inputs) 
rather than a vague "sometimes makes mistakes" statement.

